# 01: Install Python libraries

In [1]:
!pip install ollama

# 02: Update and install packages

In [2]:
!sudo apt update
!usdo apt install -y pciutils
!sudo apt install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [107 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,557 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.7 MB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,199 kB]
Get:14 http

# 03: Ollama LLMs model tester

In [8]:
from asyncio.unix_events import subprocess
import time
import ollama
import asyncio
import nest_asyncio
import os
import requests
from transformers import pipeline
import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize
import re
from typing import Optional, List, Dict, Any

nest_asyncio.apply()


class OllamaModelTester:

    def __init__(self, host, port, models):
        self.host = host
        self.port = port
        self.models = models if models else []
        self.process = None
        self.results = []
        self.dft_sleep_sec = 10
        self.nli_model = None
        self.initialization()

    def initialization(self):
        os.environ['OLLAMA_HOST'] = f'{self.host}:{self.port}'

        try:
            nltk.data.find('tokenizers/punkt_tab/english')
        except LookupError:
            nltk.download('punkt_tab')
            nltk.download('punkt')
            nltk.download('averaged_perceptron_tagger_eng')

        self.nli_model = pipeline('text-classification',
            model='cross-encoder/nli-deberta-v3-base',
            device=0
        )

        print('Class OllamaModelTester is initialized')

    def start_server(self):
        self.process = subprocess.Popen(
            ['ollama', 'serve'],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            env=os.environ
        )
        time.sleep(self.dft_sleep_sec)
        result = subprocess.run(
            ['curl', '-s', f'http://{self.host}:{self.port}/api/tags'],
            capture_output=True,
            text=True
        )
        if (result.returncode == 0):
            print('Ollama server is started')
        else:
            print('Ollama server is NOT started')

    def stop_server(self):
        self.process.terminate()
        self.process.wait()
        print('Ollama server is terminated')

    def pull_model(self, model_name):
        result = subprocess.run(
            ['ollama', 'pull', model_name],
            capture_output=True,
            text=True
        )
        if (result.returncode == 0):
            print(f'"{model_name}" model is pulled successfully!')
        else:
            print(f'"{model_name}" model pull threw an error')

    def pull_models(self, models):
        models = models if models else self.models
        for model_name in models:
            result = subprocess.run(
                ['ollama', 'pull', model_name],
                capture_output=True,
                text=True
            )
            if (result.returncode == 0):
                print(f'"{model_name}" model is pulled successfully!')
            else:
                print(f'"{model_name}" model pull threw an error')

    def compare_models(self, prompt_text, models, **options):
        var_models = models if models else self.models
        var_options = {
            'temperature': options.get('temperature', 0.1),
            'num_ctx': options.get('num_ctx', 512)
        }
        for model_name in var_models:
            try:
                print(f'Testing model "{model_name}"')
                start=time.time()

                response = ollama.generate(
                    model=model_name,
                    prompt=prompt_text,
                    options=var_options
                )
                elapsed=time.time() - start
                model_comparison: Dict[str, Any] = {
                    'model': model_name,
                    'response': response['response'],
                    'tokens': len(response['response'].split()),
                    'elapsed_time': round(elapsed, 2)
                }
                calculate_scores = self.__calculate_scores(prompt_text, response['response'])
                model_comparison.update(calculate_scores)
                self.results.append(model_comparison)
                print(f'Successfully completed testing model "{model_name}"')
            except Exception as e:
                self.results.append({
                    'model': model_name,
                    'response': f'"{model_name}" error: {str(e)}',
                    'tokens': 0,
                    'elapsed_time': 0
                })
                print(f'Testing model "{model_name}" threw an error')
        return self.results

    def print_results(self):
        for row in self.results:
            print('')
            [print(f'{k}: {v}') for k, v in row.items()]
            print('')

    def __clean_text(self, text):
        """
        Meeting 2
        Private method: Preprocess text for tokenization

        Args:
            text(str): Text to preprocess

        Returns:
            str: Preprocessed text
        """
        text = text.lower()
        text = re.sub(r'[^\w\s]', '', text)
        text = ' '.join(text.split())
        return text

    def __calculate_scores(self, reference_text, generated_text):
        """
        Meeting 2
        Private method: Calculate multiple BLUE score variants

        Args:
            reference_text(str): Text passed from main
            generated_text(str): Text generated from AI agent

        Returns:
            Dict[str, float]

        """
        dict_scores = {
            'total': 0.0,
            'factual': 0.0,
            'contradictions': 0.0,
            'neutral': 0.0,
            'confidence': 0.0,
            'is_hallucinated': 0.0,
            'bleu_1': 0.0,
            'bleu_2': 0.0,
            'bleu_3': 0.0,
            'bleu_4': 0.0,
            'bleu_avg': 0.0,
            'bleu_smoothing_method0': 0.0,
            'bleu_smoothing_method1': 0.0,
            'bleu_smoothing_method2': 0.0,
            'bleu_smoothing_method3': 0.0,
            'bleu_smoothing_method4': 0.0,
            'bleu_smoothing_method5': 0.0,
            'bleu_smoothing_method6': 0.0,
            'bleu_smoothing_method7': 0.0
        }
        try:
            if (not generated_text or len(generated_text.strip()) == 0):
                return dict_scores

            cleaned_ref = self.__clean_text(reference_text)
            cleaned_gen = self.__clean_text(generated_text)

            reference_tokens = word_tokenize(cleaned_ref)
            generated_tokens = word_tokenize(cleaned_gen)

            if (not reference_tokens or not generated_tokens):
                return dict_scores

            sentences = nltk.sent_tokenize(generated_text)
            if not sentences:
                return dict_scores

            result_scores = []
            for sentence in sentences:
                pred = self.nli_model(f'{reference_text} </s> {sentence}')
                label = (pred[0]['label']).upper()
                score = pred[0]['score']

                result_scores.append({
                    'sentence': sentence,
                    'label': label,
                    'confidence': score
                })
            print(result_scores)
            dict_scores['total'] = len(result_scores)
            dict_scores['factual'] = sum(1 for r in result_scores if r['label'] == 'ENTAILMENT') / len(result_scores)
            dict_scores['contradictions'] = sum(1 for r in result_scores if r['label'] == 'CONTRADICTION') / len(result_scores)
            dict_scores['neutral'] = sum(1 for r in result_scores if r['label'] == 'NEUTRAL') / len(result_scores)
            dict_scores['confidence'] = sum(r['confidence'] for r in result_scores) / len(result_scores)
            dict_scores['is_hallucinated'] = (dict_scores['contradictions'] > 0)

            # Calculating BLEU scores
            smoothing = SmoothingFunction()
            var_weights = (0.25, 0.25, 0.25, 0.25)

            dict_scores['bleu_1'] = sentence_bleu([reference_tokens], generated_tokens, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smoothing.method1)
            dict_scores['bleu_2'] = sentence_bleu([reference_tokens], generated_tokens, weights=(0.5, 0.5, 0.0, 0.0), smoothing_function=smoothing.method1)
            dict_scores['bleu_3'] = sentence_bleu([reference_tokens], generated_tokens, weights=(0.33, 0.33, 0.33, 0.0), smoothing_function=smoothing.method1)
            dict_scores['bleu_4'] = sentence_bleu([reference_tokens], generated_tokens, weights=(var_weights), smoothing_function=smoothing.method1)
            dict_scores['bleu_avg'] = sum([
                dict_scores['bleu_1'],
                dict_scores['bleu_2'],
                dict_scores['bleu_3'],
                dict_scores['bleu_4']
            ]) / 4.0

            # Calculating smoothing methods
            dict_scores['bleu_smoothing_method0'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method0)
            dict_scores['bleu_smoothing_method1'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method1)
            dict_scores['bleu_smoothing_method2'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method2)
            dict_scores['bleu_smoothing_method3'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method3)
            dict_scores['bleu_smoothing_method4'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method4)
            dict_scores['bleu_smoothing_method5'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method5)
            dict_scores['bleu_smoothing_method6'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method6)
            dict_scores['bleu_smoothing_method7'] = sentence_bleu([reference_tokens], generated_tokens, weights=var_weights, smoothing_function=smoothing.method7)

            return dict_scores
        except Exception as e:
            print(f'Error appeared: {str(e)}')
            return dict_scores

    def __enter__(self):
        self.start_server()
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        self.stop_server()


if (__name__ == '__main__'):
    i_models = ['tinyllama']
    i_prompt_text = """Summarize this text

Intel Corp. raised $20 billion in an upsized share sale, a third more than it was targeting when it announced the deal Monday morning.

The chipmaker priced the offering at $95 per share, according to a company statement. That represents a discount of 6.5% to Friday’s closing price, according to Bloomberg calculations. The share sale drew more than $100 billion in demand, people familiar with the matter said.
"""

    with OllamaModelTester(host='127.0.0.1', port=11434, models=i_models) as om_tester:
        om_tester.pull_models(models=None)
        om_tester.compare_models(prompt_text=i_prompt_text, models=None)
        om_tester.print_results()


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

Class OllamaModelTester is initialized
Ollama server is started
"tinyllama" model is pulled successfully!
Testing model "tinyllama"
[{'sentence': 'Intel Corp. Raised $20 billion in an upsized share sale, a third more than it was targeting when it announced the deal Monday morning.', 'label': 'ENTAILMENT', 'confidence': 0.9948364496231079}, {'sentence': 'The chipmaker priced the offering at $95 per share, representing a discount of 6.5% to Friday’s closing price, according to Bloomberg calculations.', 'label': 'NEUTRAL', 'confidence': 0.866956889629364}, {'sentence': 'The share sale drew more than $100 billion in demand, people familiar with the matter said.', 'label': 'ENTAILMENT', 'confidence': 0.9341965913772583}]
Successfully completed testing model "tinyllama"

model: tinyllama
response: Intel Corp. Raised $20 billion in an upsized share sale, a third more than it was targeting when it announced the deal Monday morning. The chipmaker priced the offering at $95 per share, representi